In [ ]:
!uv pip install langchain_gepa_adapter==0.1.0


In [ ]:
!uv pip install langchain-openrouter

In [ ]:
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
from langchain_gepa_adapter import (
    LangChainGEPAAdapter,
    last_message_text,
    make_default_proposer,
    make_reflection_lm,
    load_chat_model
)

from __future__ import annotations

import argparse
import logging
import math
import random
import re

from dotenv import load_dotenv
from gepa import optimize
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

Using Python 3.12.13 environment at: /usr
Resolved 30 packages in 1.27s
Prepared 3 packages in 95ms
Uninstalled 1 package in 10ms
Installed 5 packages in 13ms
 + jsonpath-python==1.1.6
 - langchain-core==1.3.1
 + langchain-core==1.4.0
 + langchain-openrouter==0.2.3
 + langchain-protocol==0.0.15
 + openrouter==0.9.1


In [2]:
from langchain.chat_models import init_chat_model

In [17]:
SEED_SYSTEM_PROMPT = """Add adjacent pairs of numbers, then multiply the results.
If only one pair numbers, just add the numbers

Pairs Example: 1,2
- Add pairs: 1+2=3
- multiply: Assume 1, 3*1=3
- Answer: 3

Example: 3, 5, 2, 4
- Add pairs: 3+5=8, 2+4=6
- Multiply: 8*6=48
- Answer: 48

Now solve the problem below. Put your final answer in <answer> tags. Be very concise"""



## Generate the data

A synthetic math task: add adjacent pairs, then multiply the results. Copied environment from Jonathan Whitaker (https://www.youtube.com/watch?v=yId2PE5Qmqo)



In [ ]:
def generate_problem(rng: random.Random, difficulty: str) -> dict:
    if difficulty == "easy":
        n = 4
    elif difficulty == "medium":
        n = 6
    else:
        n = 12

    nums = [
        rng.randint(1, 9) if rng.random() < 0.7 else rng.randint(10, 19)
        for _ in range(n)
    ]
    sums = [nums[i] + nums[i + 1] for i in range(0, len(nums), 2)]
    answer = math.prod(sums)

    return {
        "input": f"Numbers: {', '.join(map(str, nums))}",
        "answer": str(answer),
        "additional_context": {"nums": nums, "sums": sums},
    }


def generate_dataset(num_examples: int, difficulty: str, seed: int) -> list[dict]:
    rng = random.Random(seed)
    return [generate_problem(rng, difficulty) for _ in range(num_examples)]

In [22]:
total = 200
train_size=100
val_size = 50

all_data = generate_dataset(num_examples=total, difficulty='hard', seed=42)
train_set = all_data[: train_size]
val_set = all_data[train_size:train_size+ val_size]
test_set = all_data[train_size+val_size :]
print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")

Train: 100, Val: 50, Test: 50


## Load the chat models

openrouter models - but feel free to use anything you have access to via LangChain!

In [66]:
task_llm = init_chat_model(
  "openrouter:openai/gpt-4.1-nano",
  #reasoning={"effort": "none"}
)
reflection_llm = init_chat_model(
  "openrouter:openai/gpt-5-mini",
  reasoning={"effort": "low"}
)

In [67]:
resp = task_llm.invoke('hi')
print(resp.content)

Hello! How can I assist you today?


In [68]:
resp= reflection_llm.invoke('hi')
print(resp.content)

Hello! How can I help you today?


## Create rollout_fn

The `rollout_fn` takes a candidate item from the optimizer + example from the dataset, runs the langchain model/graph, and returns a state object. For a simple single turn chat model invoke, the state object is just the updated messages list

In [69]:
def rollout(candidate: dict[str, str], example: dict, llm: BaseChatModel) -> dict:
  messages = [
      SystemMessage(content=candidate["system_prompt"]),
      HumanMessage(content=example["input"]),
  ]
  result = llm.invoke(messages)
  if not isinstance(result, AIMessage):
      result = AIMessage(content=getattr(result, "content", str(result)))
  return {"messages": messages + [result]}

## Create eval_fn

The `eval_fn` takes as input the state object (dict) from the rollout and assigns a score/textual feedback for the optimizer

In [70]:
def evaluate_response(data: dict, state: dict) -> tuple[float, str]:
  correct_answer = data["answer"]
  nums = data["additional_context"]["nums"]
  sums = data["additional_context"]["sums"]

  response = last_message_text(state)
  match = re.search(r"<answer>\s*(\S+)\s*</answer>", response)
  if not match:
      return 0.0, (
          f"Missing <answer> tags. Could not parse answer. "
          f"The correct answer is {correct_answer}."
      )

  parsed = match.group(1).strip()
  if parsed == correct_answer.strip():
      return 1.0, "Correct."

  pairs_str = ", ".join(
      f"{nums[i]}+{nums[i+1]}={sums[i // 2]}" for i in range(0, len(nums), 2)
  )
  return 0.0, (
      f"Wrong answer: you said {parsed}, correct is {correct_answer}. "
      f"Steps: add pairs [{pairs_str}], then multiply sums to get {correct_answer}."
  )

## Create the Adapter

The adapter follows the [standard interface for GEPA adapters](https://gepa-ai.github.io/gepa/guides/adapters/) is what exposes the `evaluate()` and `make_reflective_dataset()` functions for the gepa optimizer

In [71]:
adapter = LangChainGEPAAdapter(
    rollout_fn=lambda candidate, example: rollout(candidate, example, task_llm),
    eval_fn=evaluate_response,
    num_threads=2,
    custom_proposer=make_default_proposer(make_reflection_lm(reflection_llm)),
)

## Understand your starting point/baseline

With the seed prompt, how do we perform on our test dataset?

In [73]:
print("\nBaseline evaluation on test set...")
baseline_batch = adapter.evaluate(
    batch=test_set,
    candidate={"system_prompt": SEED_SYSTEM_PROMPT},
    capture_traces=True,
)
n = len(test_set)
baseline_correct = sum(1 for s in baseline_batch.scores if s == 1.0)
baseline_acc = baseline_correct / n * 100
print(f"\nBaseline:  {baseline_correct}/{n} ({baseline_acc:.1f}%)")


Baseline evaluation on test set...

Baseline:  6/50 (12.0%)


`adapter.evaluate()` returns a `gepa.core.adapter.EvaluationBatch`, which lets you inspectthe `scores`, `outputs`, and `trajectories` of all items in the batch.

In [86]:
EXAMPLE_ID = 1

baseline_batch.scores[EXAMPLE_ID]

0.0

In [87]:
baseline_batch.outputs[EXAMPLE_ID]['state']['messages'][-1].content

'Add pairs: 4+18=22, 15+14=29, 9+4=13, 8+1=9, 2+19=21, 3+18=21  \nMultiply: 22*29*13*9*21*21 = 246,868,319,857, ending with 57.  \n<answer>57</answer>'

In [91]:
print(baseline_batch.trajectories[0].keys())
print(baseline_batch.trajectories[EXAMPLE_ID]['feedback'])

dict_keys(['data', 'state', 'score', 'feedback'])
Wrong answer: you said 57, correct is 32918886. Steps: add pairs [4+18=22, 15+14=29, 9+4=13, 8+1=9, 2+19=21, 3+18=21], then multiply sums to get 32918886.


## Run the Optimization

In [74]:
result = optimize(
    seed_candidate={"system_prompt": SEED_SYSTEM_PROMPT},
    trainset=train_set,
    valset=val_set,
    adapter=adapter,
    reflection_lm=make_reflection_lm(reflection_llm),
    max_metric_calls=500,
    reflection_minibatch_size=3,
    candidate_selection_strategy="pareto",
    use_merge=True,
    display_progress_bar=True,
    seed=0,
)

print(f"\nBest val score: {result.val_aggregate_scores[result.best_idx]}")
print("\Starting system prompt:")
print("=" * 80)
print(SEED_SYSTEM_PROMPT)
print("=" * 80)
print("\nOptimized system prompt:")
print("=" * 80)
print(result.best_candidate["system_prompt"])
print("=" * 80)



<>:16: SyntaxWarning: invalid escape sequence '\S'
<>:16: SyntaxWarning: invalid escape sequence '\S'
/tmp/ipykernel_10607/3005703388.py:16: SyntaxWarning: invalid escape sequence '\S'
  print("\Starting system prompt:")
GEPA Optimization:   0%|          | 0/500 [00:00<?, ?rollouts/s]

evaluate:   0%|          | 0/50 [00:00<?, ?it/s]

GEPA Optimization:  10%|█         | 50/500 [00:43<06:31,  1.15rollouts/s]

Iteration 0: Base program full valset score: 0.1 over 50 / 50 examples
Iteration 1: Selected program 0 score: 0.1


evaluate:   0%|          | 0/3 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Evaluate New System Prompt on Test Set

In [ ]:
print("\nOptimized evaluation on test set...")
optimized_batch = adapter.evaluate(
    batch=test_set,
    candidate=result.best_candidate,
    capture_traces=False,
)
optimized_correct = sum(1 for s in optimized_batch.scores if s == 1.0)
optimized_acc = optimized_correct / n * 100

print(f"\nBaseline:  {baseline_correct}/{n} ({baseline_acc:.1f}%)")
print(f"Optimized: {optimized_correct}/{n} ({optimized_acc:.1f}%)")
print(f"Delta:     {optimized_acc - baseline_acc:+.1f}%")